In [ ]:
# =====================================================
# Selección automática de modelo de resúmenes
# =====================================================
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
CSV_PATH = "resultados_modelos.csv"

# --- Carga tus resultados (ajusta la ruta) ---
df = pd.read_csv(CSV_PATH, sep=";", engine="python")
def to_numeric_clean(s):
    """
    Limpia texto y convierte valores numéricos, corrigiendo comas decimales.
    Ejemplo: '0,28' -> '0.28'
    """
    s = s.astype(str).str.replace(",", ".", regex=False)  # <-- corrige coma decimal
    s = s.str.replace(r"[^\d\.\-eE]", "", regex=True)     # elimina cualquier otro símbolo
    s = s.replace({"": None})
    return pd.to_numeric(s, errors="coerce")
# --- Limpia y convierte ---
num_cols = [
    "mean_alignscore", "mean_f1", "std_alignscore", "inferencia_time",
    "mean_flesch_reading_ease", "VRAM_inference_min",
    "VRAM_inference_max", "VRAM_inference_mean","mean_text_standard"
]
for c in num_cols:
    if c in df.columns:
        df[c] = to_numeric_clean(df[c])
# --- Métricas clave ---
metrics = ["mean_alignscore", "mean_f1", "std_alignscore", "mean_flesch_reading_ease", "inferencia_time"]

# --- Normalización [0,1] ---
scaler = MinMaxScaler()
df_norm = df.copy()
df_norm[metrics] = scaler.fit_transform(df[metrics])

# --- Calcula score compuesto ---
df_norm["score"] = (
    0.4 * df_norm["mean_alignscore"] # Factualidad
    + 0.40 * df_norm["mean_flesch_reading_ease"] # Claridad / legibilidad
    + 0.13 * df_norm["mean_f1"] # Coherencia/resumen
    - 0.05 * df_norm["mean_text_standard"]      # Penalizar complejidad global
    - 0.01 * df_norm["std_alignscore"]
    - 0.01 * df_norm["inferencia_time"]
)

# --- Filtros duros (ajusta según tu GPU) ---
VRAM_LIMIT = 20.0
TIME_LIMIT = 40.0

df_sel = df_norm[
    (df["VRAM_inference_max"] <= VRAM_LIMIT)
    & (df["inferencia_time"] <= TIME_LIMIT)
].copy()

# --- Ordena por score descendente ---
df_sel = df_sel.sort_values("score", ascending=False)

# --- Muestra top 3 ---
cols = ["Modelo", "mean_alignscore", "mean_f1", "std_alignscore",
        "inferencia_time", "VRAM_inference_mean", "score"]
print(df_sel[cols].head(3))

# --- Mejor modelo ---
best = df_sel.iloc[0]
print("\n=== Mejor modelo según score compuesto ===")
print(best["Modelo"], f"→ Score {best['score']:.3f}")

            Modelo  mean_alignscore   mean_f1  std_alignscore  \
9  Llama3.2-3b_COT         0.666667  0.830415             0.5   
8  Llama3.2-1b_COT         0.770833  0.844298             0.7   
5        Qwen3_COT         0.395833  0.302667             0.2   

   inferencia_time  VRAM_inference_mean     score  
9         0.132251                 3.85  0.074734  
8         0.553132                 3.85  0.020850  
5         1.000000                 4.75  0.019112  

=== Mejor modelo según score compuesto ===
Llama3.2-3b_COT → Score 0.075


In [14]:
df_sel

,Modelo,mean_precision,mean_recall,mean_f1,backbone_for_bertscore,idf,rescale_with_baseline,batch_size,device,n_examples,...,evaluation_mode,batch_size.1,device.1,ckpt_path,flag_threshold,VRAM_inference_min,VRAM_inference_max,VRAM_inference_mean,inferencia_time,score
9,Llama3.2-3b_COT,0.855951,0.851068,0.830415,roberta-base,True,False,1,cuda,380,...,nli_sp,16,cuda,models/alignscore/AlignScore-base.ckpt,"0,5",3.1,4.6,3.85,0.132251,0.665732
7,Gemma_COT,0.859264,0.847410,0.820279,roberta-base,True,False,1,cuda,380,...,nli_sp,16,cuda,models/alignscore/AlignScore-base.ckpt,"0,5",2.7,4.1,3.40,0.184223,0.628536
2,Llama3.2-3b,0.823384,0.814482,0.034950,roberta-base,True,False,1,cpu,380,...,nli_sp,16,cpu,models/alignscore/AlignScore-base.ckpt,"0,5",3.7,5.5,4.60,0.139675,0.236252
1,Llama3.2-1b,0.827922,0.813044,0.068649,roberta-base,True,False,1,cpu,379,...,nli_sp,16,cpu,models/alignscore/AlignScore-base.ckpt,"0,5",3.3,4.6,3.95,0.063109,0.229343
